# Grafici per relazione / presentazione — Macro & Micro

Set di grafici in **palette chiara accademica** (colorblind-safe) sui dataset `fineExp`, `sepsis`, `testBank`, **solo modalità isolated**.

**Tesi narrativa:** le metriche globali (macro) sono cieche perché una singola anomalia tocca poche tracce su un log enorme → l'analisi micro rivela quali anomalie sono *positive* (varianti legittime, Δ≤0) o *negative* (deviazioni da correggere, Δ>0).

I PNG vengono salvati in `Plots/` con prefisso `<dataset>_macro_1..5_*` e `<dataset>_micro_1..4_*`.

## Setup — palette, helper e loader dei dati

In [1]:
# -*- coding: utf-8 -*-
"""
Grafici relazione/presentazione — Macro & Micro (solo dati ISOLATED).
Palette chiara accademica, colorblind-safe.
"""
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

warnings.filterwarnings("ignore")

# ── Percorsi ──────────────────────────────────────────────────────────────────
ROOT     = Path(r"C:\Users\matte\Documents\GitHub\BigData-Analytics")
RESULTS  = ROOT / "results"
PLOTS    = ROOT / "notebooks" / "fineExp" / "Plots"
PLOTS.mkdir(parents=True, exist_ok=True)
DATASETS = ["fineExp", "sepsis", "testBank"]
MAX_ROWS = 26   # cap barre orizzontali per dataset molto numerosi (es. testBank, 76 anomalie)

# ── Palette chiara accademica ─────────────────────────────────────────────────
BG_FIG   = "#ffffff"
BG_AX    = "#fafafa"
TEXT     = "#1a1a1a"
MUTED    = "#6b6b6b"
GRID     = "#e6e6e6"
POS      = "#2a9d8f"   # migliora
NEG      = "#e76f51"   # peggiora
NEUTRAL  = "#adb5bd"   # invariato
BLUE     = "#264653"
AMBER    = "#e9c46a"

DIVERGING = LinearSegmentedColormap.from_list("teal_orange", [NEG, NEUTRAL, POS])

plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.serif":         ["DejaVu Serif"],
    "font.size":          10,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     10.5,
    "figure.facecolor":   BG_FIG,
    "axes.facecolor":     BG_AX,
    "axes.edgecolor":     "#cfcfcf",
    "axes.labelcolor":    TEXT,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "xtick.color":        MUTED,
    "ytick.color":        MUTED,
    "text.color":         TEXT,
    "grid.color":         GRID,
    "grid.linewidth":     0.8,
    "axes.grid":          True,
    "axes.axisbelow":     True,
    "legend.facecolor":   "#ffffff",
    "legend.edgecolor":   "#cfcfcf",
    "legend.framealpha":  0.95,
    "legend.fontsize":    9,
    "figure.dpi":         120,
    "savefig.dpi":        160,
    "savefig.facecolor":  BG_FIG,
    "savefig.bbox":       "tight",
})


def title_serif(ax, main, sub=None, accent=BLUE):
    """Titolo serif con accento colorato a sinistra."""
    ax.set_title(main, loc="left", fontfamily="DejaVu Serif",
                 fontsize=13.5, color=TEXT, pad=12 if sub is None else 20)
    if sub:
        ax.text(0.0, 1.015, sub, transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9.5, color=MUTED, fontfamily="DejaVu Sans")
    ax.spines["left"].set_color(accent)
    ax.spines["left"].set_linewidth(2.2)
    ax.spines["bottom"].set_color("#cfcfcf")


def save(fig, name):
    out = PLOTS / name
    fig.savefig(out)
    plt.close(fig)
    print(f"  -> {out.name}")


def smart_annotate(ax, xs, ys, labels, color=BLUE):
    """Etichette con bbox bianco e de-overlap verticale in coordinate display."""
    fig = ax.figure
    fig.canvas.draw()
    disp = [ax.transData.transform((x, y)) for x, y in zip(xs, ys)]
    ax_top = ax.transAxes.transform((0, 1))[1]
    order = sorted(range(len(xs)), key=lambda i: -disp[i][1])
    placed = []
    for i in order:
        px, py = disp[i]
        near_top = py > ax_top - 55            # punti vicini al bordo alto → etichetta sotto
        step = -15 if near_top else 15
        dx, dy, tries = 9, (-14 if near_top else 6), 0
        while any(abs((px + dx) - qx) < 46 and abs((py + dy) - qy) < 15
                  for qx, qy in placed) and tries < 8:
            dy += step
            tries += 1
        placed.append((px + dx, py + dy))
        ax.annotate(labels[i], (xs[i], ys[i]), textcoords="offset points",
                    xytext=(dx, dy), fontsize=8.5, color=color, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75),
                    arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6), zorder=6)


# ── Loader ────────────────────────────────────────────────────────────────────
def paths_for(ds):
    macro_candidates = [
        RESULTS / f"new_experiments_matrix_{ds}_isolated_freq_sorted.csv",
        RESULTS / f"new_experiments_matrix_{ds}_isolated.csv",
    ]
    macro = next((p for p in macro_candidates if p.exists()), None)
    return {
        "macro":   macro,
        "micro":   RESULTS / f"micro_fitness_evaluation_{ds}_isolated.csv",
        "summary": RESULTS / f"summary_{ds}_isolated.csv",
    }


def load_macro(ds):
    df = pd.read_csv(paths_for(ds)["macro"])
    df.columns = df.columns.str.strip()
    df["Sub"] = df["Scenario"].str.extract(r"(Sub\d+)", expand=False).fillna("BASELINE")
    return df


def load_micro(ds):
    return pd.read_csv(paths_for(ds)["micro"])


def build_summary(ds, save_csv=True):
    """Rigenera summary_<ds>_isolated.csv dal micro raw (Part 2 fixata, no path hardcoded)."""
    df = load_micro(ds)
    rows = []
    for anom, g in df.groupby("Anomaly_ID"):
        n = len(g)
        rows.append({
            "Anomaly_ID":        anom,
            "Num_Traces":        n,
            "Mean_Fitness_Pre":  g["Fitness_Pre"].mean(),
            "Mean_Fitness_Post": g["Fitness_Post"].mean(),
            "Pct_Improved":      round((g["Outcome"] == "IMPROVED").sum() / n * 100, 2),
            "Pct_Worsened":      round((g["Outcome"] == "WORSENED").sum() / n * 100, 2),
            "Pct_Unchanged":     round((g["Outcome"] == "UNCHANGED").sum() / n * 100, 2),
            "Mean_Delta":        g["Delta"].mean(),
            "Variance_Delta":    g["Delta"].var(),
        })
    summary = pd.DataFrame(rows)
    if save_csv:
        out = paths_for(ds)["summary"]
        summary.to_csv(out, index=False)
        print(f"  summary -> {out.name} ({len(summary)} anomalie)")
    return summary


def load_summary(ds):
    p = paths_for(ds)["summary"]
    if p.exists():
        return pd.read_csv(p)
    return build_summary(ds)


def baseline_row(macro):
    return macro[macro["Strategy"] == "BASELINE"].iloc[0]


def repairs_of(macro):
    return macro[macro["Strategy"] == "REPAIR"].copy()


## Parte 1 — Rigenerazione dei summary  (micro → summary)

Ricostruisce `summary_<dataset>_isolated.csv` da ogni file micro (versione corretta della vecchia *Part 2*, senza path hardcoded). Genera anche il summary mancante di `testBank`.

In [2]:
for ds in DATASETS:
    build_summary(ds)


  summary -> summary_fineExp_isolated.csv (22 anomalie)
  summary -> summary_sepsis_isolated.csv (27 anomalie)
  summary -> summary_testBank_isolated.csv (76 anomalie)


## Grafici MACRO (livello modello)

1. Impatto vs volume · 2. Δ Fitness · 3. Macro-vs-Micro · 4. Ventaglio baseline→repaired · 5. Δ Precision & Generalization

In [3]:
# ══════════════════════════════ MACRO ════════════════════════════════════════
def m1_impact_vs_volume(ds):
    macro = load_macro(ds)
    base = baseline_row(macro)
    rep = repairs_of(macro)
    rep["dFit"] = rep["Fitness"] - base["Fitness"]
    rep = rep[rep["Local_Modified_Traces"] > 0].copy()
    if rep.empty:
        return
    colors = [POS if d >= 0 else NEG for d in rep["dFit"]]

    fig, ax = plt.subplots(figsize=(9, 5.6))
    ax.axhline(0, color=AMBER, lw=1.6, zorder=1)
    ax.scatter(rep["Local_Modified_Traces"], rep["dFit"], s=90, c=colors,
               alpha=0.85, edgecolors="white", linewidths=0.8, zorder=3)
    ax.set_xscale("symlog")
    top = rep.nlargest(4, "Local_Modified_Traces")
    smart_annotate(ax, top["Local_Modified_Traces"].values, top["dFit"].values,
                   top["Sub"].values)
    ax.set_xlabel("Tracce modificate dal repair (scala log)")
    ax.set_ylabel("Δ Fitness macro  (repaired − baseline)")
    title_serif(ax, f"{ds} · Impatto macro vs volume di tracce",
                "Anche i repair su migliaia di tracce muovono la Fitness globale di ~0", accent=BLUE)
    save(fig, f"{ds}_macro_1_impact_vs_volume.png")


def m2_delta_fitness(ds):
    macro = load_macro(ds)
    base = baseline_row(macro)
    rep = repairs_of(macro)
    rep["dFit"] = rep["Fitness"] - base["Fitness"]
    rep = rep[rep["Local_Modified_Traces"] > 0].copy()
    if rep.empty:
        return
    capped = len(rep) > MAX_ROWS
    if capped:
        rep = rep.reindex(rep["dFit"].abs().sort_values(ascending=False).index).head(MAX_ROWS)
    rep = rep.sort_values("dFit")
    colors = [POS if d >= 0 else NEG for d in rep["dFit"]]

    h = max(4.5, 0.42 * len(rep) + 1.8)
    fig, ax = plt.subplots(figsize=(9, h))
    bars = ax.barh(rep["Sub"], rep["dFit"], color=colors, alpha=0.9,
                   edgecolor="white", linewidth=0.6, zorder=3)
    lim = rep["dFit"].abs().max() * 1.35 or 0.001
    off = lim * 0.02
    for b, v in zip(bars, rep["dFit"]):
        ax.text(b.get_width() + (off if v >= 0 else -off), b.get_y() + b.get_height()/2,
                f"{v:+.4f}", va="center", ha="left" if v >= 0 else "right",
                fontsize=8, color=TEXT)
    ax.axvline(0, color=AMBER, lw=1.4, zorder=4)
    ax.set_xlim(-lim, lim)
    ax.set_xlabel("Δ Fitness macro  (repaired − baseline)")
    ax.grid(axis="y", visible=False)
    sub = "Esclusi i repair no-op (0 tracce modificate)"
    if capped:
        sub = f"Prime {MAX_ROWS} anomalie per |Δ| · esclusi i repair no-op"
    title_serif(ax, f"{ds} · Δ Fitness per anomalia (solo Sub con effetto)", sub, accent=POS)
    save(fig, f"{ds}_macro_2_delta_fitness.png")


def m3_macro_vs_micro(ds):
    macro = load_macro(ds)
    base = baseline_row(macro)
    rep = repairs_of(macro)
    rep["dFit_macro"] = rep["Fitness"] - base["Fitness"]
    summ = load_summary(ds).rename(columns={"Anomaly_ID": "Sub"})
    m = rep.merge(summ[["Sub", "Mean_Delta", "Num_Traces"]], on="Sub", how="inner")
    m = m[m["Num_Traces"] > 0].copy()
    if m.empty:
        return
    colors = [POS if d >= 0 else NEG for d in m["Mean_Delta"]]
    sizes = 40 + 160 * (m["Num_Traces"] / m["Num_Traces"].max())

    fig, ax = plt.subplots(figsize=(8.6, 6))
    ax.axhline(0, color=MUTED, lw=0.9, ls="--", zorder=1)
    ax.axvline(0, color=MUTED, lw=0.9, ls="--", zorder=1)
    ax.scatter(m["Mean_Delta"], m["dFit_macro"], s=sizes, c=colors, alpha=0.8,
               edgecolors="white", linewidths=0.8, zorder=3)
    top = m.reindex(m["Mean_Delta"].abs().sort_values(ascending=False).index).head(5)
    smart_annotate(ax, top["Mean_Delta"].values, top["dFit_macro"].values, top["Sub"].values)
    ax.set_xlabel("Δ Fitness MICRO  (media per traccia)")
    ax.set_ylabel("Δ Fitness MACRO  (modello globale)")
    title_serif(ax, f"{ds} · Il macro nasconde il micro",
                "Δ micro fino a ±0.15 collassano in un macro ≈ 0  (size ∝ n. tracce)", accent=AMBER)
    save(fig, f"{ds}_macro_3_macro_vs_micro.png")


def m4_fan_baseline_repaired(ds):
    macro = load_macro(ds)
    base = baseline_row(macro)
    rep = repairs_of(macro)
    metrics = ["Fitness", "Precision", "Generalization", "Simplicity"]

    fig, axes = plt.subplots(1, 4, figsize=(12, 5))
    rng = np.random.default_rng(0)
    for ax, met in zip(axes, metrics):
        vals = rep[met].values
        x = rng.normal(0, 0.05, size=len(vals))
        ax.scatter(x, vals, s=45, c=BLUE, alpha=0.45,
                   edgecolors="white", linewidths=0.5, zorder=3, label="repair isolati")
        ax.axhline(base[met], color=AMBER, lw=2.2, zorder=4, label="baseline")
        ax.set_xlim(-0.35, 0.35)
        ax.set_xticks([])
        ax.set_title(met, fontsize=11, color=TEXT, fontfamily="DejaVu Serif")
        ax.grid(axis="x", visible=False)
        # padding y attorno ai dati per mostrare la compattezza
        lo = min(vals.min(), base[met]); hi = max(vals.max(), base[met])
        pad = (hi - lo) * 0.6 + 1e-4
        ax.set_ylim(lo - pad, hi + pad)
    axes[0].set_ylabel("Valore metrica")
    axes[-1].legend(loc="lower right", fontsize=8)
    fig.tight_layout(rect=[0, 0, 1, 0.86])
    fig.text(0.012, 0.975, f"{ds} · Metriche globali: baseline vs tutti i repair isolati",
             ha="left", fontfamily="DejaVu Serif", fontsize=13.5, fontweight="bold")
    fig.text(0.012, 0.925, "Le metriche del modello restano incollate alla baseline",
             ha="left", fontsize=9.5, color=MUTED)
    save(fig, f"{ds}_macro_4_fan_baseline_repaired.png")


def m5_delta_prec_gen(ds):
    macro = load_macro(ds)
    base = baseline_row(macro)
    rep = repairs_of(macro)
    rep = rep[rep["Local_Modified_Traces"] > 0].copy()
    if rep.empty:
        return
    n_rows = min(len(rep), MAX_ROWS)
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4.5, 0.4*n_rows+1.8)), sharey=False)
    for ax, met in zip(axes, ["Precision", "Generalization"]):
        d = rep.copy()
        d["delta"] = d[met] - base[met]
        if len(d) > MAX_ROWS:
            d = d.reindex(d["delta"].abs().sort_values(ascending=False).index).head(MAX_ROWS)
        d = d.sort_values("delta")
        colors = [POS if v >= 0 else NEG for v in d["delta"]]
        ax.barh(d["Sub"], d["delta"], color=colors, alpha=0.9,
                edgecolor="white", linewidth=0.6, zorder=3)
        ax.axvline(0, color=AMBER, lw=1.3, zorder=4)
        lim = d["delta"].abs().max() * 1.3 or 0.001
        ax.set_xlim(-lim, lim)
        ax.set_xlabel(f"Δ {met}")
        ax.grid(axis="y", visible=False)
        title_serif(ax, met, accent=BLUE)
    fig.suptitle(f"{ds} · Δ Precision & Generalization per anomalia",
                 x=0.012, ha="left", fontfamily="DejaVu Serif",
                 fontsize=13.5, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save(fig, f"{ds}_macro_5_delta_prec_gen.png")


## Grafici MICRO (livello traccia / anomalia)

1. Mappa di classificazione · 2. Composizione outcome · 3. Mean Δ-fitness · 4. Distribuzioni trace-level

In [4]:
# ══════════════════════════════ MICRO ════════════════════════════════════════
def mic1_classification_map(ds):
    s = load_summary(ds)
    s = s[s["Num_Traces"] > 0].copy()
    if s.empty:
        return
    norm = TwoSlopeNorm(vmin=0, vcenter=50, vmax=100)
    fig, ax = plt.subplots(figsize=(9.2, 6.2))
    ax.axvspan(-1, 0, color=NEG, alpha=0.05, zorder=0)
    ax.axvspan(0, 1, color=POS, alpha=0.05, zorder=0)
    ax.axvline(0, color=MUTED, lw=1.0, ls="--", zorder=1)
    sc = ax.scatter(s["Mean_Delta"], s["Num_Traces"], c=s["Pct_Improved"],
                    cmap=DIVERGING, norm=norm, s=110, alpha=0.9,
                    edgecolors="white", linewidths=0.8, zorder=3)
    ax.set_yscale("symlog")
    xlim = max(0.02, s["Mean_Delta"].abs().max() * 1.25)
    ax.set_xlim(-xlim, xlim)
    ax.text(0.25, 0.965, "← variante legittima\n(anomalia positiva)", transform=ax.transAxes,
            ha="center", va="top", fontsize=9, color=NEG, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.8))
    ax.text(0.75, 0.965, "correzione utile →\n(anomalia negativa)", transform=ax.transAxes,
            ha="center", va="top", fontsize=9, color=POS, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.8))
    top = s.reindex(s["Mean_Delta"].abs().sort_values(ascending=False).index).head(6)
    smart_annotate(ax, top["Mean_Delta"].values, top["Num_Traces"].values,
                   top["Anomaly_ID"].values)
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label("% tracce migliorate", fontsize=9)
    ax.set_xlabel("Δ Fitness medio per traccia")
    ax.set_ylabel("N. tracce interessate (scala log)")
    title_serif(ax, f"{ds} · Mappa di classificazione delle anomalie",
                "Positive (variante) vs negative (da correggere) secondo l'effetto sulla conformità",
                accent=AMBER)
    save(fig, f"{ds}_micro_1_classification_map.png")


def mic2_outcome_composition(ds):
    s = load_summary(ds).copy()
    s = s[s["Num_Traces"] > 0].copy()
    if s.empty:
        return
    capped = len(s) > MAX_ROWS
    if capped:
        s = s.nlargest(MAX_ROWS, "Num_Traces")
    s["net"] = s["Pct_Improved"] - s["Pct_Worsened"]
    s = s.sort_values("net")
    h = max(4.5, 0.34 * len(s) + 1.8)
    fig, ax = plt.subplots(figsize=(9.5, h))
    y = np.arange(len(s))
    ax.barh(y, s["Pct_Improved"], color=POS, label="Migliorate", zorder=3)
    ax.barh(y, s["Pct_Unchanged"], left=s["Pct_Improved"], color=NEUTRAL,
            label="Invariate", zorder=3)
    ax.barh(y, s["Pct_Worsened"], left=s["Pct_Improved"] + s["Pct_Unchanged"],
            color=NEG, label="Peggiorate", zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(s["Anomaly_ID"], fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Composizione degli outcome per traccia (%)", labelpad=8)
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.045 - 1.2/h),
              ncol=3, frameon=False)
    sub = f"Prime {MAX_ROWS} anomalie per n. tracce" if capped else None
    title_serif(ax, f"{ds} · Esito del repair traccia-per-traccia", sub, accent=BLUE)
    save(fig, f"{ds}_micro_2_outcome_composition.png")


def mic3_mean_delta(ds):
    s = load_summary(ds).copy()
    s = s[(s["Num_Traces"] > 0) & (s["Pct_Unchanged"] < 100)].copy()
    if s.empty:
        return
    capped = len(s) > MAX_ROWS
    if capped:
        s = s.reindex(s["Mean_Delta"].abs().sort_values(ascending=False).index).head(MAX_ROWS)
    s["std"] = np.sqrt(s["Variance_Delta"].fillna(0.0))
    s = s.sort_values("Mean_Delta")
    colors = [POS if v >= 0 else NEG for v in s["Mean_Delta"]]
    h = max(4.5, 0.4 * len(s) + 1.8)
    fig, ax = plt.subplots(figsize=(9.5, h))
    y = np.arange(len(s))
    ax.barh(y, s["Mean_Delta"], xerr=s["std"], color=colors, alpha=0.9,
            edgecolor="white", linewidth=0.6, zorder=3,
            error_kw=dict(ecolor=MUTED, elinewidth=1, capsize=2))
    ax.axvline(0, color=AMBER, lw=1.4, zorder=4)
    ax.set_yticks(y)
    ax.set_yticklabels(s["Anomaly_ID"], fontsize=8)
    lim = (s["Mean_Delta"].abs() + s["std"]).max() * 1.25
    ax.set_xlim(-lim, lim)
    for yi, (_, r) in zip(y, s.iterrows()):
        ax.text(lim * 0.98, yi, f"n={int(r['Num_Traces'])}", va="center", ha="right",
                fontsize=7.5, color=MUTED)
    ax.set_xlabel("Δ Fitness medio per traccia (± dev. std)")
    ax.grid(axis="y", visible=False)
    sub = "Δ>0 correzione utile · Δ<0 variante legittima"
    if capped:
        sub = f"Prime {MAX_ROWS} anomalie per |Δ| · Δ>0 correzione utile, Δ<0 variante legittima"
    title_serif(ax, f"{ds} · Verdetto quantitativo per anomalia", sub, accent=POS)
    save(fig, f"{ds}_micro_3_mean_delta.png")


def mic4_delta_distributions(ds):
    micro = load_micro(ds)
    counts = micro.groupby("Anomaly_ID").size()
    keep = counts[counts >= 10].sort_values(ascending=False).head(12).index.tolist()
    if not keep:
        return
    data = [micro.loc[micro["Anomaly_ID"] == a, "Delta"].values for a in keep]
    medians = [np.median(d) for d in data]
    order = np.argsort(medians)
    keep = [keep[i] for i in order]
    data = [data[i] for i in order]
    medians = [medians[i] for i in order]

    h = max(4.5, 0.5 * len(keep) + 1.8)
    fig, ax = plt.subplots(figsize=(9.5, h))
    pos = np.arange(len(keep))
    for i, (d, med) in enumerate(zip(data, medians)):
        col = POS if med >= 0 else NEG
        if np.ptp(d) < 1e-9:  # distribuzione degenere
            ax.plot([d[0]], [i], marker="D", ms=8, color=col, zorder=4)
        else:
            vp = ax.violinplot(d, positions=[i], vert=False, widths=0.8,
                               showextrema=False)
            for b in vp["bodies"]:
                b.set_facecolor(col); b.set_alpha(0.35); b.set_edgecolor(col)
            bp = ax.boxplot(d, positions=[i], vert=False, widths=0.25,
                            patch_artist=True, showfliers=False)
            for box in bp["boxes"]:
                box.set(facecolor="white", edgecolor=col, linewidth=1.2)
            for w in bp["whiskers"] + bp["caps"]:
                w.set(color=col, linewidth=1.0)
            for m in bp["medians"]:
                m.set(color=BLUE, linewidth=1.6)
    ax.axvline(0, color=AMBER, lw=1.4, zorder=1)
    ax.set_yticks(pos)
    ax.set_yticklabels(keep, fontsize=8.5)
    ax.set_xlabel("Δ Fitness per traccia")
    ax.grid(axis="y", visible=False)
    title_serif(ax, f"{ds} · Distribuzione dell'effetto sulle tracce",
                "Anomalie con ≥10 tracce · effetto omogeneo vs disperso", accent=BLUE)
    save(fig, f"{ds}_micro_4_delta_distributions.png")


## Esecuzione — genera tutti i PNG in `Plots/`

In [5]:
# ── main ──────────────────────────────────────────────────────────────────────
MACRO = [m1_impact_vs_volume, m2_delta_fitness, m3_macro_vs_micro,
         m4_fan_baseline_repaired, m5_delta_prec_gen]
MICRO = [mic1_classification_map, mic2_outcome_composition,
         mic3_mean_delta, mic4_delta_distributions]


def main():
    for ds in DATASETS:
        print(f"\n=== {ds} ===")
        # assicura il summary (genera testBank)
        if not paths_for(ds)["summary"].exists():
            build_summary(ds)
        for fn in MACRO + MICRO:
            try:
                fn(ds)
            except Exception as e:
                print(f"  [skip] {fn.__name__}: {e}")


main()



=== fineExp ===


  -> fineExp_macro_1_impact_vs_volume.png
  -> fineExp_macro_2_delta_fitness.png


  -> fineExp_macro_3_macro_vs_micro.png


  -> fineExp_macro_4_fan_baseline_repaired.png


  -> fineExp_macro_5_delta_prec_gen.png


...

  -> testBank_macro_5_delta_prec_gen.png


  -> testBank_micro_1_classification_map.png


  -> testBank_micro_2_outcome_composition.png


  -> testBank_micro_3_mean_delta.png


  -> testBank_micro_4_delta_distributions.png
